# Configuração da IA no Google Colab
Execute as células abaixo para configurar o Ollama com o modelo de visão e expor a API usando o Cloudflare Tunnel.

In [1]:
# 1. Monta o Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
drive_ollama_path = '/content/drive/MyDrive/ollama_models'
os.makedirs(drive_ollama_path, exist_ok=True)

!rm -rf /root/.ollama
!mkdir -p /root/
!ln -s {drive_ollama_path} /root/.ollama
print("✅ Drive configurado!")

# 2. Instala dependências (incluindo pyngrok)
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q fastapi uvicorn pydantic requests pyngrok
print("\n✅ Dependências instaladas!")


Mounted at /content/drive
✅ Drive configurado!
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,806 kB]
Get:12 https://ppa.launchpadconte

KeyboardInterrupt: 

In [ ]:
import subprocess
import time

print("Iniciando Ollama em background...")
subprocess.Popen(["ollama", "serve"])
time.sleep(3) 

# Baixando a versão Vision do Qwen (leva uns minutinhos na primeira vez)
print("Baixando o modelo de visão (qwen2.5-vl)...")
!ollama pull qwen2.5-vl
print("✅ Qwen-VL pronto para uso!")


Iniciando Ollama em background...
Verificando/Baixando o modelo de visão (llava)... Isso leva cerca de 2 minutinhos.

✅ Modelo pronto!


In [ ]:
import threading
import uvicorn
import requests
import time
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

# 🔴 O SEU TOKEN DO NGROK AQUI
NGROK_AUTH_TOKEN = "3GBRe0mUquxhlqrudg8d2RgHh9s_3qNPk8vmi363x9LB78g4G"

app = FastAPI()

class RequestData(BaseModel):
    prompt: str
    image: str # Imagem em base64 (sem data:image/jpeg;base64,)

@app.get("/")
def home():
    return {"status": "online", "message": "Servidor Qwen2.5-VL rodando com Ngrok!"}

@app.post("/analyze")
async def analyze_part(data: RequestData):
    ollama_url = "http://localhost:11434/api/generate"
    
    # ⬇️ Alterado o modelo para qwen2.5-vl aqui também!
    payload = {
        "model": "qwen2.5-vl", 
        "prompt": data.prompt + "\n\nResponda ESTRITAMENTE em formato JSON com as chaves: is_car_part (boolean), title (string), brand (string), category (string).",
        "images": [data.image],
        "stream": False,
        "format": "json"
    }
    
    try:
        response = requests.post(ollama_url, json=payload)
        response_data = response.json()
        resultado_json = response_data.get("response", "{}")
        return {"content": resultado_json}
    except Exception as e:
        return {"error": str(e)}

def start_ngrok():
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill() # Limpa sessões antigas
    
    tunnel = ngrok.connect(8000)
    url = tunnel.public_url
    
    print("\n" + "="*60)
    print(f"✅ QWEN-VL ONLINE COM NGROK!")
    print(f"➡️ Para testar no navegador: {url}")
    print(f"➡️ URL FINAL (Bot/Supabase): {url}/analyze")
    print("="*60 + "\n")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

print("Ligando os motores do servidor...")
threading.Thread(target=run_server, daemon=True).start()

time.sleep(3) 
start_ngrok()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nEncerrando...")
    ngrok.kill()


Iniciando servidores...


INFO:     Started server process [354]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
